# after run dw.py, map ids to uniport

In [2]:
import pandas as pd
import mygene

def get_map_df(ensembl_ids,input_type):
    mg = mygene.MyGeneInfo()
    # Query mygene for UniProt and Entrez gene ID mappings
    results = mg.querymany(
        ensembl_ids,
        scopes=input_type,
        fields='uniprot,entrezgene',
        species='human'
    )

    results_df = pd.DataFrame(results)
    results_df = results_df[~results_df['entrezgene'].isna()]
    results_df['uniprot_ids'] = results_df['uniprot'].apply(
        lambda x: list(x.values())[0] if isinstance(x, dict) and 'Swiss-Prot' in x else None)
    results_df = results_df[~results_df['uniprot_ids'].isna()]
    results_df[results_df['uniprot_ids'].apply(lambda x: isinstance(x, list) and len(x) > 1)]
    return results_df

In [7]:
file_path = '/itf-fi-ml/shared/users/ziyuzh/svm/data/biograd/biograd_entrz_2019_dw_emb_40.txt'

ppi_emb = pd.read_csv(file_path,sep='\s+', skiprows=1, header=None)
ppi_emb.columns = ['string_id'] + [f'feature_{i}' for i in range(1, len(ppi_emb.columns))]

ppi_ids_map = get_map_df(ppi_emb['string_id'],'entrezgene')

ppi_set = set()
for values in ppi_ids_map['uniprot_ids']:
    if isinstance(values, list) and len(values) > 1:
        ppi_set.update(values)  # Add all elements in the list
    else:
        ppi_set.add(values)

string_ids = []
one2more = []
more2one = []  # to collect subdfs with multiple or zero matches

for uniport_ids in list(ppi_set):
    subdf = ppi_ids_map[ppi_ids_map['uniprot_ids'].str.contains(uniport_ids, na=False)]
    
    if len(subdf) == 1:
        if isinstance(subdf['uniprot_ids'], list) and len(values) > 1:
            one2more.append(subdf)
        else:
            string_ids.append(uniport_ids)
    else:
        more2one.append(subdf)

more2one_df = pd.concat(more2one, ignore_index=True)

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
54 input query terms found no hit:	['101929876', '26148', '11217', '23285', '554223', '114299', '348738', '101060521', '100506627', '10


In [24]:
file_path = '/itf-fi-ml/shared/users/ziyuzh/svm/data/biograd/biograd_entrz_2019_dw_emb_40.txt'

ppi_emb = pd.read_csv(file_path,sep='\s+', skiprows=1, header=None)
ppi_emb.columns = ['string_id'] + [f'feature_{i}' for i in range(1, len(ppi_emb.columns))]

In [25]:
ppi_emb["string_id"] = ppi_emb["string_id"].astype(str)
ppi_emb = ppi_emb.set_index("string_id")


# Prepare list to store the results
aggregated_rows = []

# Iterate over each UniProt ID group
for protein_id, subdf in more2one_df.groupby('uniprot_ids'):
    # Get list of ENSP IDs
    ensp_ids = subdf['query'].tolist()

    # Select corresponding rows from ppi_emb where 'string_id' is in ensp_ids
    matched_ppi = ppi_emb[ppi_emb.index.isin(ensp_ids)]

    if not matched_ppi.empty:
        # Calculate the mean of all feature columns (exclude 'string_id')
        mean_features = matched_ppi.mean()

        # Create a new row with UniProt ID and the averaged features
        mean_features['string_id'] = protein_id

        # Add to the results list
        aggregated_rows.append(mean_features)

In [28]:


# Convert the list of Series into a DataFrame
aggregated_df = pd.DataFrame(aggregated_rows)

# Optional: Reorder columns to have 'uniprot_id' first
cols = ['string_id'] + [col for col in aggregated_df.columns if col != 'string_id']
aggregated_df = aggregated_df[cols]

# Step 1: Prepare ENSP IDs with '9606.' prefix
ensp_ids = [ensp_id for ensp_id in ppi_ids_map[ppi_ids_map['uniprot_ids'].isin(string_ids)]['query'].tolist()]

# Step 2: Select matching rows from ppi_emb
other_ppi = ppi_emb[ppi_emb.index.isin(ensp_ids)].copy()

# Step 3: Map 'string_id' back to 'uniprot_ids'
ensp_to_uniprot = ppi_ids_map[ppi_ids_map['uniprot_ids'].isin(string_ids)].set_index('query')['uniprot_ids'].to_dict()

# Apply mapping to refill 'string_id' with corresponding UniProt ID
other_ppi['string_id'] = other_ppi.index.map(lambda x: ensp_to_uniprot[x])

bio_emb_df = pd.concat([other_ppi, aggregated_df], ignore_index=True)



In [29]:
bio_emb_df

,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,...,feature_120,feature_121,feature_122,feature_123,feature_124,feature_125,feature_126,feature_127,feature_128,string_id
0,0.056083,-0.023629,0.056343,0.308701,-0.172503,-0.278329,-0.150651,-0.116942,-0.394997,-0.153239,...,-0.033820,-0.008002,-0.409115,0.001032,-0.180018,0.020003,-0.234411,-0.064717,0.241991,Q14258
1,0.158077,-0.135066,0.263737,0.046041,0.001912,-0.072776,0.185127,0.238026,-0.138225,0.342231,...,-0.126026,-0.128634,-0.137477,0.170881,0.419760,-0.345539,-0.035867,-0.528335,-0.224792,P05067
2,-0.025345,-0.392939,0.031069,0.121674,0.244977,-0.178394,-0.237143,-0.108559,-0.383149,0.054730,...,-0.004881,-0.327615,0.134563,-0.201960,0.189251,-0.038664,0.195635,-0.169837,0.264428,Q92731
3,0.092364,0.115733,0.265615,0.170219,0.001859,-0.112354,-0.135642,-0.084671,0.007350,-0.057334,...,-0.088765,-0.263335,-0.185058,0.022640,0.032358,-0.130116,-0.326183,-0.095681,0.091486,P04629
4,-0.068925,-0.255730,0.273689,-0.200226,-0.297978,-0.173201,-0.030832,0.311092,-0.612800,0.102472,...,0.252174,-0.162515,0.033788,0.040758,0.068632,-0.116704,-0.436574,-0.407268,-0.307009,Q15717
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16191,-0.063660,0.071476,0.385776,0.269819,0.054358,-0.459274,-0.543911,0.130654,-0.263577,0.187696,...,0.194278,-0.287211,-0.128094,0.159143,0.195725,-0.270593,-0.394281,-0.043897,0.036268,Q9UEU5
16192,0.231067,-0.170838,0.675367,-0.129552,0.367257,-0.404887,-0.169539,0.165464,0.013575,0.075758,...,-0.049563,0.186830,-0.510623,-0.557297,0.367508,0.003910,-0.779987,-0.161066,-0.033530,Q9ULR0
16193,-0.163749,0.108507,0.019783,0.169763,0.190702,-0.642601,-0.047960,0.177299,0.179134,-0.550418,...,0.077291,-0.370575,-0.441849,-0.556600,-0.076311,0.098187,0.049798,-0.149377,-0.019784,Q9Y3E7
16194,0.119962,-0.510105,-0.219148,-0.203809,-0.362119,-0.284135,0.029741,0.382501,-0.371610,0.144513,...,0.291605,0.019331,-0.972366,-0.391732,0.232158,-0.022988,-0.257245,-0.088305,0.213705,Q9Y6F7


In [30]:
bio_emb_df.to_csv(file_path,index = False)


In [31]:
file_path

'/itf-fi-ml/shared/users/ziyuzh/svm/data/biograd/biograd_entrz_2019_dw_emb_40.txt'

In [33]:
n2v = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/biograd/uniport_biogrid_emb_2019.csv')

In [36]:
len(set(n2v['string_id'].tolist()) | set(bio_emb_df['string_id'].tolist()))

16196